In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("1. Εκκίνηση αυτόνομης SparkSession για GBT...")
spark = SparkSession.builder \
    .appName("GBT_SMOTE_GridSearch") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("2. Φόρτωση Δεδομένων (Απευθείας από το SMOTE Gold dataset)...")
train_gold = spark.read.parquet("../../data/train_gold_with_cluster.parquet")
test_gold = spark.read.parquet("../../data/test_gold_with_cluster.parquet")

# Μετατροπές τύπων για ασφάλεια
train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))
train_gold = train_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))
test_gold = test_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))

print("3. Feature Augmentation (Ενσωμάτωση K-Means Cluster)...")
assembler = VectorAssembler(inputCols=["features", "cluster"], outputCol="augmented_features")

train_augmented = assembler.transform(train_gold).drop("features").withColumnRenamed("augmented_features", "features")
test_augmented = assembler.transform(test_gold).drop("features").withColumnRenamed("augmented_features", "features")

# Caching του πλήρους SMOTE dataset (ΑΠΑΡΑΙΤΗΤΟ για το GBT που κάνει πολλές επαναλήψεις)
train_augmented.cache()
test_augmented.cache()

print("4. Ορισμός GBT και Πλέγματος Παραμέτρων (Grid Search)...")
# Σημείωση: Το GBT του PySpark δεν παίρνει weightCol, αλλά δεν μας νοιάζει γιατί έχουμε SMOTE (50-50)!
gbt_base = GBTClassifier(featuresCol="features", labelCol="stroke", seed=22390225)

# Το κλειδί στο GBT: Ρηχά δέντρα (maxDepth 3, 4, 5), και δοκιμή στον αριθμό των iterations
paramGrid = (ParamGridBuilder()
             .addGrid(gbt_base.maxDepth, [3, 4, 5]) 
             .addGrid(gbt_base.maxIter, [50, 100, 150])
             .build())

# Χρησιμοποιούμε την PR-AUC για αξιολόγηση της ισορροπίας Precision/Recall
evaluator = BinaryClassificationEvaluator(
    labelCol="stroke", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderPR"
)

cv = CrossValidator(estimator=gbt_base,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=3,
                    seed=22390225)

print("5. Εκτέλεση Cross-Validation στο Πλήρες SMOTE Dataset (Θα πάρει μερικά λεπτά)...")
cv_model = cv.fit(train_augmented)
best_gbt = cv_model.bestModel

print("\n" + "="*60)
print("[ΕΥΡΕΣΗ ΠΑΡΑΜΕΤΡΩΝ GBT ΟΛΟΚΛΗΡΩΘΗΚΕ]")
print(f" -> Βέλτιστο maxDepth: {best_gbt._java_obj.getMaxDepth()}")
print(f" -> Βέλτιστο maxIter (αριθμός δέντρων): {best_gbt._java_obj.getMaxIter()}")
print("="*60 + "\n")

print("6. Παραγωγή προβλέψεων στο άγνωστο Test Set...")
gbt_preds = best_gbt.transform(test_augmented)

print("7. Αποθήκευση των τελικών προβλέψεων...")
output_path = "../../data/gbt_predictions.parquet"
gbt_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet(output_path)

print(f"===========================================================")
print(f"[ΕΠΙΤΥΧΙΑ] Το βέλτιστο GBT εκπαιδεύτηκε και αποθηκεύτηκε!")
print(f"Αρχείο: {output_path}")
print(f"===========================================================")

spark.stop()

1. Εκκίνηση αυτόνομης SparkSession για GBT...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 22:07:16 WARN Utils: Your hostname, cachyos-x8664, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface enp4s0)
26/06/09 22:07:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 22:07:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 22:07:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


2. Φόρτωση Δεδομένων (Απευθείας από το SMOTE Gold dataset)...
3. Feature Augmentation (Ενσωμάτωση K-Means Cluster)...
4. Ορισμός GBT και Πλέγματος Παραμέτρων (Grid Search)...
5. Εκτέλεση Cross-Validation στο Πλήρες SMOTE Dataset (Θα πάρει μερικά λεπτά)...


26/06/09 22:09:31 WARN DAGScheduler: Broadcasting large task binary with size 1001.8 KiB
26/06/09 22:09:31 WARN DAGScheduler: Broadcasting large task binary with size 1002.3 KiB
26/06/09 22:09:31 WARN DAGScheduler: Broadcasting large task binary with size 1003.3 KiB
26/06/09 22:09:31 WARN DAGScheduler: Broadcasting large task binary with size 1004.0 KiB
26/06/09 22:09:31 WARN DAGScheduler: Broadcasting large task binary with size 1006.4 KiB
26/06/09 22:09:31 WARN DAGScheduler: Broadcasting large task binary with size 1009.1 KiB
26/06/09 22:09:31 WARN DAGScheduler: Broadcasting large task binary with size 1009.6 KiB
26/06/09 22:09:31 WARN DAGScheduler: Broadcasting large task binary with size 1010.6 KiB
26/06/09 22:09:32 WARN DAGScheduler: Broadcasting large task binary with size 1011.4 KiB
26/06/09 22:09:32 WARN DAGScheduler: Broadcasting large task binary with size 1013.7 KiB
26/06/09 22:09:32 WARN DAGScheduler: Broadcasting large task binary with size 1016.4 KiB
26/06/09 22:09:32 WAR


[ΕΥΡΕΣΗ ΠΑΡΑΜΕΤΡΩΝ GBT ΟΛΟΚΛΗΡΩΘΗΚΕ]
 -> Βέλτιστο maxDepth: 5
 -> Βέλτιστο maxIter (αριθμός δέντρων): 150

6. Παραγωγή προβλέψεων στο άγνωστο Test Set...
7. Αποθήκευση των τελικών προβλέψεων...


26/06/09 22:14:29 WARN DAGScheduler: Broadcasting large task binary with size 1365.5 KiB


[ΕΠΙΤΥΧΙΑ] Το βέλτιστο GBT εκπαιδεύτηκε και αποθηκεύτηκε!
Αρχείο: ../../data/gbt_predictions.parquet
